# Bean Leaf Legion Classification

In [1]:
import kagglehub
import os
import shutil

target_dir = os.path.expanduser("~/Desktop/LEARNINGPANDAS/10_Pytorch/data/bean_leaf_legion")

download_path = kagglehub.dataset_download("marquis03/bean-leaf-lesions-classification")

os.makedirs(os.path.dirname(target_dir), exist_ok=True)

if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

shutil.move(download_path, target_dir)

print(f"Dataset successfully moved to: {target_dir}")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 155M/155M [01:00<00:00, 2.69MB/s] 

Extracting files...


Dataset successfully moved to: /Users/atharvakhandelwal/Desktop/LEARNINGPANDAS/10_Pytorch/data/bean_leaf_legion


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import pandas as pd
from torchvision import models
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import os

In [2]:
device = torch.device('mps')

In [3]:
train_df = pd.read_csv('/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/bean_leaf_legion/train.csv')
val_df = pd.read_csv('/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/bean_leaf_legion/val.csv')

In [4]:
train_df.head()

,image:FILE,category
0,train/healthy/healthy_train.98.jpg,0
1,train/healthy/healthy_train.148.jpg,0
2,train/healthy/healthy_train.306.jpg,0
3,train/healthy/healthy_train.305.jpg,0
4,train/healthy/healthy_train.40.jpg,0


In [5]:
train_df['image:FILE'] = '/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/bean_leaf_legion/' + train_df['image:FILE']
val_df['image:FILE'] = '/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/bean_leaf_legion/' + val_df['image:FILE']

In [6]:
train_df.iloc[0,0]

'/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/bean_leaf_legion/train/healthy/healthy_train.98.jpg'

In [7]:
val_df.iloc[0,0]

'/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/bean_leaf_legion/val/healthy/healthy_val.25.jpg'

In [8]:
transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop((224,224)),
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float),
    transforms.Normalize((0.485, 0.456, 0.406),(0.229, 0.224, 0.225))
])

In [9]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe
        self.transform = transform
        self.label = torch.tensor(dataframe["category"])

    def __len__(self):
        return self.dataframe.shape[0]

    def __getitem__(self, index):
        imgpath = self.dataframe.iloc[index,0]
        label = self.label[index]
        image = Image.open(imgpath)
        if self.transform:
            image = self.transform(image)
        return image, label

In [10]:
train_dataset = CustomDataset(train_df, transform)
test_dataset = CustomDataset(val_df, transform)

In [11]:
train_dataset[0]

(tensor([[[-0.9877, -1.1418, -1.6727,  ..., -1.2274, -1.1932, -1.1075],
          [-1.2445, -1.4329, -1.7412,  ..., -1.2103, -1.1247, -1.1075],
          [-1.4500, -1.6042, -1.6898,  ..., -1.2274, -1.1589, -1.1589],
          ...,
          [ 1.1015,  1.1358,  1.1700,  ..., -1.7754, -1.7412, -1.7069],
          [ 1.1015,  1.1015,  1.1187,  ..., -1.7412, -1.7240, -1.7069],
          [ 1.1700,  1.0673,  1.1187,  ..., -1.7069, -1.7240, -1.7069]],
 
         [[-0.4951, -0.6176, -1.2304,  ..., -0.4251, -0.3725, -0.3025],
          [-0.8277, -1.0028, -1.3529,  ..., -0.3375, -0.2500, -0.2675],
          [-1.1429, -1.2479, -1.3354,  ..., -0.3725, -0.2675, -0.3200],
          ...,
          [ 0.4328,  0.4503,  0.4678,  ..., -1.6856, -1.6506, -1.5980],
          [ 0.4153,  0.3978,  0.3803,  ..., -1.6681, -1.6506, -1.5805],
          [ 0.4678,  0.3452,  0.3803,  ..., -1.6681, -1.6331, -1.5805]],
 
         [[-1.0376, -1.2293, -1.7173,  ..., -1.7522, -1.7870, -1.7870],
          [-1.3513, -1.5256,

In [12]:
lr = 1e-3
batch_size = 32
epochs = 15

In [13]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [14]:
googlenet_model = models.googlenet(weights = 'DEFAULT')

In [15]:
for param in googlenet_model.parameters():
    param.requires_grad = True

In [16]:
classes = len(train_df["category"].unique())
classes

3

In [17]:
googlenet_model.fc = torch.nn.Linear(googlenet_model.fc.in_features, classes)
googlenet_model.to(device)

GoogLeNet(
  (conv1): BasicConv2d(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): BasicConv2d(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv3): BasicConv2d(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (inception3a): Inception(
    (branch1): BasicConv2d(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(googlenet_model.parameters(), lr=lr) 

In [21]:
from tqdm import tqdm

for epoch in range(epochs):
    googlenet_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progressbar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", leave=True)

    for input, label in progressbar:
        input = input.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        output = googlenet_model(input)

        loss = criterion(output, label)

        loss.backward()
        optimizer.step()

        batch_size = input.size(0)
        running_loss += loss.item() * batch_size
        _, preds = torch.max(output, 1)
        correct += (preds == label).sum().item()
        total += batch_size

        progressbar.set_postfix({
            "loss": f"{running_loss / total:.4f}",
            "acc": f"{correct / total:.4f}"
        })

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {epoch_loss:.4f} "
          f"Acc: {epoch_acc:.4f}")

Epoch [1/15]: 100%|██████████| 33/33 [00:33<00:00,  1.01s/it, loss=0.3701, acc=0.8588]


Epoch [1/15] Loss: 0.3701 Acc: 0.8588


Epoch [2/15]: 100%|██████████| 33/33 [00:19<00:00,  1.67it/s, loss=0.1153, acc=0.9652]


Epoch [2/15] Loss: 0.1153 Acc: 0.9652


Epoch [3/15]: 100%|██████████| 33/33 [00:19<00:00,  1.67it/s, loss=0.0702, acc=0.9778]


...

Epoch [13/15] Loss: 0.0381 Acc: 0.9845


Epoch [14/15]: 100%|██████████| 33/33 [00:19<00:00,  1.65it/s, loss=0.0305, acc=0.9884]


Epoch [14/15] Loss: 0.0305 Acc: 0.9884


Epoch [15/15]: 100%|██████████| 33/33 [00:19<00:00,  1.66it/s, loss=0.0300, acc=0.9894]

Epoch [15/15] Loss: 0.0300 Acc: 0.9894


In [22]:
googlenet_model.eval()
test_loss = 0.0
test_correct = 0
test_total = 0

progress_bar = tqdm(test_loader, desc="Validation")

with torch.no_grad():
    for inputs, labels in progress_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = googlenet_model(inputs)
        loss = criterion(outputs, labels)

        batch_size = inputs.size(0)
        test_loss += loss.item() * batch_size
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == labels).sum().item()
        test_total += batch_size

        progress_bar.set_postfix({
            "loss": f"{test_loss / test_total:.4f}",
            "acc": f"{test_correct / test_total:.4f}"
        })

test_loss /= test_total
test_acc = test_correct / test_total


Validation: 100%|██████████| 5/5 [00:02<00:00,  2.26it/s, loss=0.1652, acc=0.9624]
